In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.dataset_download("nih-chest-xrays/sample")
print("Dataset downloaded to:", path)

100%|██████████| 4.20G/4.20G [00:38<00:00, 118MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/nih-chest-xrays/sample/versions/4


In [3]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
img_dir_candidates = [root for root, dirs, files in os.walk(path) if any(f.lower().endswith(".png") for f in files)]

print("CSV found:", csv_candidates)
print("Image dirs found:", img_dir_candidates)

CSV found: ['/root/.cache/kagglehub/datasets/nih-chest-xrays/sample/versions/4/sample_labels.csv', '/root/.cache/kagglehub/datasets/nih-chest-xrays/sample/versions/4/sample/sample_labels.csv']
Image dirs found: ['/root/.cache/kagglehub/datasets/nih-chest-xrays/sample/versions/4/sample/images', '/root/.cache/kagglehub/datasets/nih-chest-xrays/sample/versions/4/sample/sample/images']


In [4]:
import pandas as pd

labels_csv_path = csv_candidates[0]
IMAGES_DIR = img_dir_candidates[0]

labels_df = pd.read_csv(labels_csv_path)
print(labels_df.shape)

all_labels = labels_df["Finding Labels"].str.split("|").explode()
label_counts = all_labels.value_counts()
print(label_counts)
print()
print(f"No Finding: {(labels_df['Finding Labels'] == 'No Finding').mean():.1%} of images")

(5606, 11)
Finding Labels
No Finding            3044
Infiltration           967
Effusion               644
Atelectasis            508
Nodule                 313
Mass                   284
Pneumothorax           271
Consolidation          226
Pleural_Thickening     176
Cardiomegaly           141
Emphysema              127
Edema                  118
Fibrosis                84
Pneumonia               62
Hernia                  13
Name: count, dtype: int64

No Finding: 54.3% of images


In [5]:
import numpy as np

CONDITIONS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

# Catches a casing/naming mismatch BEFORE training on silently-wrong labels.
missing = [c for c in CONDITIONS if c not in label_counts.index]
if missing:
    print("WARNING: these condition names don't match the dataset's labels:", missing)
    print("Available labels:", sorted(label_counts.index.tolist()))
else:
    print("All 14 condition names match the dataset's label vocabulary.")

def labels_to_vector(finding_labels_str):
    labels = finding_labels_str.split("|")
    return np.array([1.0 if c in labels else 0.0 for c in CONDITIONS], dtype=np.float32)

label_matrix = np.stack(labels_df["Finding Labels"].apply(labels_to_vector).to_numpy())
print(label_matrix.shape)

for i, c in enumerate(CONDITIONS):
    print(f"{c}: {int(label_matrix[:, i].sum())} positive examples")

All 14 condition names match the dataset's label vocabulary.
(5606, 14)
Atelectasis: 508 positive examples
Cardiomegaly: 141 positive examples
Effusion: 644 positive examples
Infiltration: 967 positive examples
Mass: 284 positive examples
Nodule: 313 positive examples
Pneumonia: 62 positive examples
Pneumothorax: 271 positive examples
Consolidation: 226 positive examples
Edema: 118 positive examples
Emphysema: 127 positive examples
Fibrosis: 84 positive examples
Pleural_Thickening: 176 positive examples
Hernia: 13 positive examples


In [6]:
from sklearn.model_selection import train_test_split

image_indices = labels_df["Image Index"].to_numpy()
train_idx, val_idx = train_test_split(np.arange(len(image_indices)), test_size=0.2, random_state=42)

train_files = image_indices[train_idx]
val_files = image_indices[val_idx]
train_labels = label_matrix[train_idx]
val_labels = label_matrix[val_idx]

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

Train: 4484, Val: 1122


In [7]:
import tensorflow as tf

IMG_SIZE = 128

def load_image(filename, label):
    path = tf.strings.join([IMAGES_DIR, filename], separator="/")
    image = tf.io.read_file(path)
    image = tf.io.decode_png(image, channels=3)  # handles grayscale X-rays -> RGB automatically
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, label

def make_dataset(files, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if shuffle:
        ds = ds.shuffle(len(files), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, train_labels, shuffle=True)
val_ds = make_dataset(val_files, val_labels)

# confirm a batch actually loads before committing to a full training run
for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "batch label shape:", labs.shape)

batch image shape: (32, 128, 128, 3) batch label shape: (32, 14)


In [8]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CONDITIONS), activation="sigmoid"),  # NOT softmax — multi-label
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │         1,806 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,631,150 (6.22 MB)

 Trainable params: 1,631,150 (6.22 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

callbacks = [EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    shuffle=False,  # dataset is already shuffled above; avoids a harmless-but-noisy warning
    callbacks=callbacks,
)

Epoch 1/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 81s 523ms/step - auc: 0.4889 - loss: 0.2099 - val_auc: 0.5287 - val_loss: 0.1818
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 64s 457ms/step - auc: 0.5207 - loss: 0.1895 - val_auc: 0.5745 - val_loss: 0.1792
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 55s 390ms/step - auc: 0.5656 - loss: 0.1839 - val_auc: 0.5854 - val_loss: 0.1836
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 91s 457ms/step - auc: 0.5995 - loss: 0.1787 - val_auc: 0.5984 - val_loss: 0.1789
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 83s 466ms/step - auc: 0.6318 - loss: 0.1751 - val_auc: 0.6132 - val_loss: 0.1776
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 65s 463ms/step - auc: 0.6538 - loss: 0.1717 - val_auc: 0.6027 - val_loss: 0.1776
Epoch 7/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 81s 457ms/step - auc: 0.6877 - loss: 0.1682 - val_auc: 0.6347 - val_loss: 0.1783
Epoch 8/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 82s 455ms/step - auc: 0.7208 - loss: 0.1643 - val_auc: 0.6139 - val_loss: 0.1815
Epoch 9/30
141/141 ━━━━━━━━━━━━━

In [10]:
from sklearn.metrics import roc_auc_score

y_true = val_labels
y_pred = model.predict(val_ds, verbose=0)

print(f"{'Condition':<20} {'AUC':>6} {'Positives in val':>18}")
aucs = []
for i, c in enumerate(CONDITIONS):
    n_pos = int(y_true[:, i].sum())
    if n_pos == 0 or n_pos == len(y_true):
        print(f"{c:<20} {'N/A':>6} {n_pos:>18}  (too few/no positive examples to compute AUC)")
        continue
    auc = roc_auc_score(y_true[:, i], y_pred[:, i])
    aucs.append(auc)
    print(f"{c:<20} {auc:>6.3f} {n_pos:>18}")

print(f"\nMean AUC across conditions with enough data: {np.mean(aucs):.3f}")

Condition               AUC   Positives in val
Atelectasis           0.650                120
Cardiomegaly          0.579                 23
Effusion              0.688                128
Infiltration          0.620                189
Mass                  0.575                 55
Nodule                0.572                 58
Pneumonia             0.679                 12
Pneumothorax          0.579                 50
Consolidation         0.696                 50
Edema                 0.801                 31
Emphysema             0.619                 27
Fibrosis              0.513                 21
Pleural_Thickening    0.600                 25
Hernia                0.343                  3

Mean AUC across conditions with enough data: 0.608


In [11]:
import json

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v1_baseline_cnn.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

Artifacts written:
total 19176
drwxr-xr-x 2 root root     4096 Sep 21 09:50 .
drwxr-xr-x 1 root root     4096 Sep 21 09:50 ..
-rw-r--r-- 1 root root      219 Sep 21 09:50 condition_names.json
-rw-r--r-- 1 root root 19621620 Sep 21 09:50 v1_baseline_cnn.keras


In [12]:
import shutil

shutil.make_archive("v1_imaging_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v1_imaging_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>